# 🎲 Naive Bayes Sınıflandırıcısı: Ayrıntılı Teorik Anlatım ve Uygulama Rehberi

Bu çalışma kitabında, olasılık kuramına dayanan hızlı, anlaşılır ve etkili bir sınıflandırma algoritması olan **Naive Bayes** hem matematiksel/teorik temelleriyle ele alınmakta hem de `ayakkabi_numarasi.xlsx` veri kümesi üzerinde **Cinsiyet Tahmini (Sınıflandırma)** uygulamasıyla adım adım gerçekleştirilmektedir.

---

## 📚 Bölüm 1: Naive Bayes Nedir ve Nasıl Çalışır?

**Naive Bayes**, Thomas Bayes tarafından geliştirilen **Bayes Teoremi**'ne dayanan olasılıksal (*probabilistic*) bir makine öğrenmesi algoritmasıdır. Bir gözlemin hangi sınıfa ait olduğunu bulmak için, eldeki özelliklerin her bir sınıfta görülme olasılıklarını çarparak en yüksek olasılıklı sınıfı seçer.

### 1.1. Bayes Teoreminin Matematiksel Formülü
Gözlemlenen özellik vektörü $X = (x_1, x_2, \dots, x_n)$ verildiğinde, bu örneğin $C_k$ sınıfına (örneğin `Erkek` veya `Kadın`) ait olma olasılığı şöyle hesaplanır:

$$P(C_k \mid X) = \frac{P(X \mid C_k) \cdot P(C_k)}{P(X)}$$

Buradaki terimlerin anlamları:
1. **$P(C_k \mid X)$ — Sonsal Olasılık (*Posterior Probability*):** Özellikleri ($X$: Boy, Kilo, Yaş, Ayakkabı No) bilinen bir kişinin $C_k$ sınıfına ait olma olasılığıdır (Aradığımız nihai olasılık).
2. **$P(C_k)$ — Önsel Olasılık (*Prior Probability*):** Hiçbir özelliğe bakmadan, eğitim verisinde $C_k$ sınıfının genel bulunma oranıdır (Örn: Eğitim setinin %54'ü Erkek, %46'sı Kadın).
3. **$P(X \mid C_k)$ — Olabilirlik (*Likelihood*):** Sınıfın $C_k$ olduğu bilindiğinde, bu $X$ özelliklerinin gözlemlenme olasılığıdır.
4. **$P(X)$ — Kanıt (*Evidence*):** Özelliklerin genel görülme olasılığıdır. Tüm sınıflar için ortak payda olduğundan karar aşamasında sabit kabul edilir.

--- 

## 🤔 Bölüm 2: Neden "Naive" (Saf / İyimser) Denir?

Gerçek dünyada özellikler birbirine bağlı olabilir (örneğin **Boy** arttıkça **Kilo** ve **Ayakkabı Numarası** da artma eğilimindedir). Ancak birleşik olasılık $P(x_1, x_2, \dots, x_n \mid C_k)$ hesabını doğrudan yapmak çok zordur.

Algoritma burada **"Naive" (Saf)** bir varsayımda bulunur: **Tüm bağımsız değişkenlerin ($x_1, x_2, \dots, x_n$) birbirinden tamamen bağımsız olduğunu varsayar.** Bu sayede birleşik olasılık, tekil olasılıkların basit bir çarpımına dönüşür:

$$P(X \mid C_k) = P(x_1 \mid C_k) \cdot P(x_2 \mid C_k) \cdots P(x_n \mid C_k) = \prod_{i=1}^{n} P(x_i \mid C_k)$$

Sınıflandırıcı en yüksek olasılığı veren sınıfı ($\hat{y}$) seçer:

$$\hat{y} = \arg\max_{C_k} \left( P(C_k) \prod_{i=1}^{n} P(x_i \mid C_k) \right)$$

---

## 📈 Bölüm 3: Naive Bayes Türleri ve Gaussian Naive Bayes

Veri tipine göre 3 temel Naive Bayes modeli vardır:
1. **Gaussian Naive Bayes (`GaussianNB`):** Özellikler **sürekli sayısal** değerler olduğunda (Boy, Kilo, Yaş gibi) kullanılır. Her özelliğin sınıf içinde **Normal (Gauss) Dağılım (Çan Eğrisi)** gösterdiğini varsayar.
2. **Multinomial Naive Bayes (`MultinomialNB`):** Metin sınıflandırmada kelime frekansları (sayma verileri) için kullanılır.
3. **Bernoulli Naive Bayes (`BernoulliNB`):** Özelliklerin `0` ve `1` gibi ikili (var/yok) olduğu durumlarda kullanılır.

### Gaussian Naive Bayes Formülü
Bizim veri setimizdeki `Kilo`, `Boy`, `Yaş` ve `Ayakkabı Numarası` sürekli sayısal değişkenler olduğundan **`GaussianNB`** kullanacağız. Model eğitim aşamasında her sınıf ($C_k$) ve her özellik ($x_i$) için sadece **Ortalama ($\mu_{k,i}$)** ve **Varyans ($\sigma^2_{k,i}$)** hesaplar:

$$P(x_i \mid C_k) = \frac{1}{\sqrt{2\pi\sigma^2_{k,i}}} \exp\left( -\frac{(x_i - \mu_{k,i})^2}{2\sigma^2_{k,i}} \right)$$

- **Önemli Özellik:** Her değişken kendi ortalama ve standart sapmasıyla Gauss formülüne girdiği için, Naive Bayes algoritması `StandardScaler` gibi bir ölçeklendirmeye **ihtiyaç duymaz**, çok hızlı eğitilir ve küçük veri setlerinde aşırı öğrenmeye (*overfitting*) karşı oldukça dirençlidir.

## Adım 1: Gerekli Kütüphanelerin Yüklenmesi
Veri analizi, görselleştirme ve `sklearn.naive_bayes` altındaki `GaussianNB` sınıfını içe aktarıyoruz.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Grafik ayarları
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)

## Adım 2: Veri Setinin Yüklenmesi ve Gauss (Normal) Dağılımlarının İncelenmesi
`ayakkabi_numarasi.xlsx` dosyasını yüklüyor ve `GaussianNB` algoritmasının temelini oluşturan sınıf bazlı olasılık yoğunluk (çan eğrisi) dağılımlarını görselleştiriyoruz.

In [ ]:
# Excel dosyasını okuma
df = pd.read_excel('ayakkabi_numarasi.xlsx')

# Hedef değişkeni ikili sayısal formata kodlama (Erkek: 1, Kadın: 0)
df['Cinsiyet_E'] = (df['Cinsiyet(E veya K)'] == 'E').astype(int)

print(f"Veri Boyutu: {df.shape[0]} gözlem, {df.shape[1]} sütun")
display(df.head())

In [ ]:
# Özelliklerin Kadın ve Erkek sınıflarındaki Gauss (KDE) dağılımlarını görselleştirme
feature_cols = ['Kilo(kg)', 'Boy(metre)', 'Yaş', 'Ayakkabı Numarası']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    sns.kdeplot(data=df, x=col, hue='Cinsiyet(E veya K)', fill=True, common_norm=False, palette='Set1', alpha=0.4, ax=axes[i])
    axes[i].set_title(f"{col} - Cinsiyete Göre Olasılık Yoğunluğu")

plt.tight_layout()
plt.show()

## Adım 3: Değişkenlerin Ayrılması ve Eğitim / Test Bölünmesi
- **Bağımsız Değişkenler ($X$):** `Kilo(kg)`, `Boy(metre)`, `Yaş`, `Ayakkabı Numarası`
- **Bağımlı Hedef Değişken ($y$):** `Cinsiyet_E` (`1: Erkek`, `0: Kadın`)
- Diğer modellerimizle birebir karşılaştırabilmek için aynı `%84 Eğitim - %16 Test` (`random_state=42`) ayrımını kullanıyoruz.

In [ ]:
X = df[feature_cols]
y = df['Cinsiyet_E']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.16, random_state=42
)

print(f"Eğitim seti boyutu: {X_train.shape[0]} örnek")
print(f"Test seti boyutu:   {X_test.shape[0]} örnek")

## Adım 4: Gaussian Naive Bayes Modelinin Eğitilmesi ve Öğrenilen Parametreler
Modeli eğitiyoruz ve `GaussianNB`'nin eğitim verisinden öğrendiği:
1. **Önsel Olasılıkları (`class_prior_`):** $P(\text{Kadın})$ ve $P(\text{Erkek})$
2. **Sınıf Ortalamalarını (`theta_`):** $\mu_{k,i}$
3. **Sınıf Varyanslarını (`var_`):** $\sigma^2_{k,i}$ 
tablo halinde inceliyoruz.

In [ ]:
# Gaussian Naive Bayes modelinin oluşturulması ve eğitilmesi
gnb_model = GaussianNB()
gnb_model.fit(X_train, y_train)

print("Gaussian Naive Bayes modeli başarıyla eğitildi!\n")

# 1. Önsel Olasılıklar (Priors)
print(f"Önsel Olasılıklar P(C) -> Kadın (0): %{gnb_model.class_prior_[0]*100:.2f}, Erkek (1): %{gnb_model.class_prior_[1]*100:.2f}\n")

# 2. Modelin öğrendiği özellik ortalamaları (Mu)
df_means = pd.DataFrame(gnb_model.theta_, index=['Kadın (0)', 'Erkek (1)'], columns=feature_cols)
print("--- Modelin Öğrendiği Sınıf Ortalamaları (Mu) ---")
display(df_means.round(3))

# 3. Modelin öğrendiği özellik standart sapmaları (Sigma)
df_std = pd.DataFrame(np.sqrt(gnb_model.var_), index=['Kadın (0)', 'Erkek (1)'], columns=feature_cols)
print("\n--- Modelin Öğrendiği Sınıf Standart Sapmaları (Sigma) ---")
display(df_std.round(3))

## Adım 5: Modelin Değerlendirilmesi (Test Kümesi Performansı)
Test verisi üzerinde tahminler üreterek **Doğruluk (*Accuracy*)**, **Sınıflandırma Raporu** ve **Karmaşıklık Matrisi (*Confusion Matrix*)** sonuçlarını inceliyoruz.

In [ ]:
# Test seti tahminleri
y_pred = gnb_model.predict(X_test)

# Doğruluk (Accuracy) skoru
acc = accuracy_score(y_test, y_pred)
print(f"Test Seti Doğruluk Oranı (Accuracy): %{acc * 100:.2f}\n")

# Detaylı Sınıflandırma Raporu
print("--- Sınıflandırma Raporu ---")
print(classification_report(y_test, y_pred, target_names=['Kadın (0)', 'Erkek (1)']))

# Karmaşıklık Matrisi Görselleştirmesi
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4.5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Purples',
    xticklabels=['Kadın (0)', 'Erkek (1)'],
    yticklabels=['Kadın (0)', 'Erkek (1)']
)
plt.xlabel('Tahmin Edilen Sınıf')
plt.ylabel('Gerçek Sınıf')
plt.title('Gaussian Naive Bayes Karmaşıklık Matrisi')
plt.show()

## Adım 6: Örnek Girdilerle Yeni Tahmin ve Sonsal Olasılıkların (`predict_proba`) İncelenmesi
Naive Bayes yalnızca sınıf etiketi (`0` veya `1`) döndürmekle kalmaz, aynı zamanda Bayes teoremiyle hesaplanan **Sonsal Olasılıkları ($P(C_k \mid X)$)** doğrudan verir.

In [ ]:
# Örnek yeni bireyler: [Kilo(kg), Boy(metre), Yaş, Ayakkabı Numarası]
yeni_ornekler = pd.DataFrame([
    {'Kilo(kg)': 78, 'Boy(metre)': 1.76, 'Yaş': 30, 'Ayakkabı Numarası': 43},
    {'Kilo(kg)': 56, 'Boy(metre)': 1.63, 'Yaş': 25, 'Ayakkabı Numarası': 37}
])

# Sınıf tahminleri ve olasılıklar
sinif_tahminleri = gnb_model.predict(yeni_ornekler)
olasiliklar = gnb_model.predict_proba(yeni_ornekler)

for i, row in yeni_ornekler.iterrows():
    tahmin_etiket = 'Erkek (1)' if sinif_tahminleri[i] == 1 else 'Kadın (0)'
    kadin_olasilik = olasiliklar[i][0] * 100
    erkek_olasilik = olasiliklar[i][1] * 100
    print(f"Birey {i+1} (Boy: {row['Boy(metre)']}m, Kilo: {row['Kilo(kg)']}kg, Yaş: {row['Yaş']}, Ayakkabı No: {row['Ayakkabı Numarası']})\n"
          f" -> Tahmin Edilen Cinsiyet: {tahmin_etiket}\n"
          f" -> Bayes Sonsal Olasılıkları: Kadın P(K|X) = %{kadin_olasilik:.2f} | Erkek P(E|X) = %{erkek_olasilik:.2f}\n")